# MyDigitalTwin — Netflix
**Notebook 01 — Ingestion, exploration, nettoyage → Parquet**

Source : `data/raw/NETFLIX/NetflixViewingHistory.csv`  
Output : `data/parquet/netflix_views.parquet`

## Objectif ML
Construire la matrice d'interactions pour l'algorithme ALS (recommandation).  
Chaque ligne = un visionnage = un signal d'intérêt.

## 0. Initialisation Spark

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '../../..')))
from config import RAW_DATA, WAREHOUSE

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DateType

spark = SparkSession.builder \
    .appName("MyDigitalTwin - Netflix") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

Spark version : 3.5.5


26/03/27 16:00:14 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 1. Ingestion

In [2]:
RAW_PATH    = os.path.join(RAW_DATA, "NETFLIX", "NetflixViewingHistory.csv")
PARQUET_OUT = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "..", "data", "parquet", "netflix_views.parquet")

# Lecture brute
df_raw = spark.read \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .csv(RAW_PATH)

print(f"Lignes brutes : {df_raw.count():,}")
df_raw.printSchema()
df_raw.show(5, truncate=60)

Lignes brutes : 4,288
root
 |-- Title: string (nullable = true)
 |-- Date: string (nullable = true)

+-----------------------------------+-------+
|                              Title|   Date|
+-----------------------------------+-------+
|Black Mirror: Saison 4: Hang the DJ|3/26/26|
|       Derrière nos écrans de fumée|3/26/26|
|                Le Monde après nous|3/25/26|
|           Les Dinosaures: La chute|3/15/26|
|           Les Dinosaures: L'empire|3/13/26|
+-----------------------------------+-------+
only showing top 5 rows



## 2. Exploration — Data Quality

In [3]:
# Valeurs nulles
print("=== Valeurs nulles ===")
df_raw.select([
    F.count(F.when(F.col(c).isNull() | (F.col(c) == ""), c)).alias(c)
    for c in df_raw.columns
]).show()

# Période couverte
print("\n=== Période ===")
df_raw.agg(
    F.min("Date").alias("premier_visionnage"),
    F.max("Date").alias("dernier_visionnage"),
    F.countDistinct("Date").alias("nb_jours_distincts")
).show()

# Titres uniques
print(f"\nTitres distincts : {df_raw.select('Title').distinct().count():,}")

=== Valeurs nulles ===
+-----+----+
|Title|Date|
+-----+----+
|    0|   0|
+-----+----+


=== Période ===


+------------------+------------------+------------------+
|premier_visionnage|dernier_visionnage|nb_jours_distincts|
+------------------+------------------+------------------+
|            1/1/22|            9/9/25|              1104|
+------------------+------------------+------------------+


Titres distincts : 4,256


In [4]:
# Exemples de titres pour comprendre les patterns
print("=== Exemples de formats de titres ===")
df_raw.select("Title") \
    .where(F.col("Title").contains(":")) \
    .distinct() \
    .limit(15) \
    .show(truncate=80)

print("\n=== Titres sans ':' (films) ===")
df_raw.select("Title") \
    .where(~F.col("Title").contains(":")) \
    .distinct() \
    .limit(10) \
    .show(truncate=80)

=== Exemples de formats de titres ===


+----------------------------------------------------------------+
|                                                           Title|
+----------------------------------------------------------------+
|                     La Traque dans le sang: Saison 1: Épisode 4|
|                  Rick et Morty: Saison 3: Repos et Ricklaxation|
|      Le Monde incroyable de Gumball: Saison 2: Le voleur / Noël|
|          Hunter X Hunter (2011): Saison 5: Ce jour, cet instant|
|Hunter X Hunter (2011): Saison 5: Retrouvailles et compréhension|
|                                    The Mist: La salle d'attente|
|             BoJack Horseman: Saison 6: Des coûts irrécupérables|
|                             Peaky Blinders: Saison 3: Épisode 5|
|                                 Sweet Home: Saison 1: Épisode 1|
|                                     Trivia Quest: De dix en dix|
|                   Love, Death & Robots: Volume 2: Le géant noyé|
|                          Breaking Bad: Saison 2: Nouvelle do

## 3. Nettoyage & enrichissement

Le format du titre Netflix suit le pattern :  
- Film : `Titre du film`  
- Série : `Nom de la série: Saison X: Titre de l'épisode`  

On parse ça pour extraire : `show_title`, `season`, `episode_title`, `content_type`.

In [5]:
# Parser la date Netflix (format M/D/YY)
df = df_raw.withColumn(
    "watch_date",
    F.to_date(F.col("Date"), "M/d/yy")
)

# Champs temporels
df = df \
    .withColumn("watch_year",    F.year("watch_date")) \
    .withColumn("watch_month",   F.date_format("watch_date", "yyyy-MM")) \
    .withColumn("watch_weekday", F.dayofweek("watch_date")) \
    .withColumn("watch_week",    F.weekofyear("watch_date"))

df.select("Title", "watch_date", "watch_year", "watch_month", "watch_weekday").show(5)

+--------------------+----------+----------+-----------+-------------+
|               Title|watch_date|watch_year|watch_month|watch_weekday|
+--------------------+----------+----------+-----------+-------------+
|Black Mirror: Sai...|2026-03-26|      2026|    2026-03|            5|
|Derrière nos écra...|2026-03-26|      2026|    2026-03|            5|
| Le Monde après nous|2026-03-25|      2026|    2026-03|            4|
|Les Dinosaures: L...|2026-03-15|      2026|    2026-03|            1|
|Les Dinosaures: L...|2026-03-13|      2026|    2026-03|            6|
+--------------------+----------+----------+-----------+-------------+
only showing top 5 rows



In [6]:
# Détecter série vs film
df = df.withColumn(
    "content_type",
    F.when(F.col("Title").contains(":"), F.lit("series")).otherwise(F.lit("movie"))
)

# Titre de la série (avant le premier ':')
df = df.withColumn(
    "show_title",
    F.when(
        F.col("content_type") == "series",
        F.trim(F.split(F.col("Title"), ":")[0])
    ).otherwise(F.col("Title"))
)

# Saison — regex sur Title (pas encore renommé)
df = df.withColumn(
    "season",
    F.when(
        F.col("content_type") == "series",
        F.regexp_extract(F.col("Title"), r"Saison.(\d+)", 1)
    ).otherwise(F.lit(""))
).withColumn(
    "season",
    F.when(F.col("season") == "", F.lit(None).cast("int"))
     .otherwise(F.col("season").cast("int"))
)

# Épisode (dernière partie après le dernier ':')
df = df.withColumn(
    "episode_title",
    F.when(
        F.col("content_type") == "series",
        F.trim(F.element_at(F.split(F.col("Title"), ":"), -1))
    ).otherwise(F.lit(None))
)

df = df.withColumn("interaction_weight", F.lit(1.0))

# Rename à la toute fin — après tous les withColumn qui utilisent "Title"
df = df.drop("Date").withColumnRenamed("Title", "raw_title")

df.select("show_title", "season", "episode_title", "content_type", "watch_date").show(10, truncate=50)

+----------------------------+------+-----------------------+------------+----------+
|                  show_title|season|          episode_title|content_type|watch_date|
+----------------------------+------+-----------------------+------------+----------+
|                Black Mirror|     4|            Hang the DJ|      series|2026-03-26|
|Derrière nos écrans de fumée|  NULL|                   NULL|       movie|2026-03-26|
|         Le Monde après nous|  NULL|                   NULL|       movie|2026-03-25|
|              Les Dinosaures|  NULL|               La chute|      series|2026-03-15|
|              Les Dinosaures|  NULL|               L'empire|      series|2026-03-13|
|              Les Dinosaures|  NULL|            La conquête|      series|2026-03-13|
|                   One Piece|     2|Les cerisiers d'Hiluluk|      series|2026-03-12|
|                   One Piece|     2|              Charlatan|      series|2026-03-12|
|                   One Piece|     2|     Un château e

## 4. Exploration après nettoyage

In [7]:
print("=== Répartition films vs séries ===")
df.groupBy("content_type").count().show()

print("\n=== Top 15 séries les plus regardées ===")
df.filter(F.col("content_type") == "series") \
  .groupBy("show_title") \
  .count() \
  .orderBy(F.desc("count")) \
  .limit(15) \
  .show(truncate=50)

print("\n=== Top 10 films les plus regardés ===")
df.filter(F.col("content_type") == "movie") \
  .groupBy("show_title") \
  .count() \
  .orderBy(F.desc("count")) \
  .limit(10) \
  .show(truncate=50)

=== Répartition films vs séries ===
+------------+-----+
|content_type|count|
+------------+-----+
|       movie|  354|
|      series| 3934|
+------------+-----+


=== Top 15 séries les plus regardées ===
+------------------------+-----+
|              show_title|count|
+------------------------+-----+
|        Naruto Shippuden|  467|
|                  Naruto|  220|
|  Hunter X Hunter (2011)|  148|
|              Fairy Tail|  146|
|JoJo's Bizarre Adventure|  109|
|   The Seven Deadly Sins|   91|
|            Regular Show|   80|
|                  Boruto|   80|
|         BoJack Horseman|   76|
|     Fullmetal Alchemist|   70|
|           Rick et Morty|   61|
|    Saiki Kusuo no Ψ Nan|   56|
|         Solar Opposites|   53|
|            Prison Break|   49|
|            Désenchantée|   49|
+------------------------+-----+


=== Top 10 films les plus regardés ===
+--------------------+-----+
|          show_title|count|
+--------------------+-----+
|                    |   19|
|         J

In [8]:
print("=== Visionnages par année ===")
df.groupBy("watch_year") \
  .count() \
  .orderBy("watch_year") \
  .show()

print("\n=== Visionnages par jour de la semaine ===")
# 1=dimanche, 2=lundi, ..., 7=samedi dans PySpark
jours = {1: "Dimanche", 2: "Lundi", 3: "Mardi", 4: "Mercredi",
         5: "Jeudi", 6: "Vendredi", 7: "Samedi"}

df.groupBy("watch_weekday") \
  .count() \
  .orderBy("watch_weekday") \
  .show()

=== Visionnages par année ===
+----------+-----+
|watch_year|count|
+----------+-----+
|      2019|  388|
|      2020|  800|
|      2021|  315|
|      2022|  278|
|      2023|  935|
|      2024|  618|
|      2025|  741|
|      2026|  213|
+----------+-----+


=== Visionnages par jour de la semaine ===
+-------------+-----+
|watch_weekday|count|
+-------------+-----+
|            1|  791|
|            2|  545|
|            3|  490|
|            4|  659|
|            5|  478|
|            6|  584|
|            7|  741|
+-------------+-----+



In [9]:
print("=== Activité mensuelle ===")
df.groupBy("watch_month") \
  .count() \
  .orderBy("watch_month") \
  .show(30)

print("\n=== Pic de visionnage (semaine la plus active) ===")
df.groupBy("watch_year", "watch_week") \
  .count() \
  .orderBy(F.desc("count")) \
  .limit(5) \
  .show()

=== Activité mensuelle ===
+-----------+-----+
|watch_month|count|
+-----------+-----+
|    2019-05|    3|
|    2019-06|   78|
|    2019-07|   22|
|    2019-08|   52|
|    2019-09|   72|
|    2019-10|   44|
|    2019-11|   53|
|    2019-12|   64|
|    2020-01|   73|
|    2020-02|   34|
|    2020-03|   66|
|    2020-04|   57|
|    2020-05|   69|
|    2020-06|   90|
|    2020-07|  111|
|    2020-08|   72|
|    2020-09|   65|
|    2020-10|   89|
|    2020-11|   64|
|    2020-12|   10|
|    2021-01|    3|
|    2021-02|   13|
|    2021-03|    5|
|    2021-04|   19|
|    2021-05|   80|
|    2021-06|    9|
|    2021-07|   39|
|    2021-08|   35|
|    2021-09|   31|
|    2021-10|    3|
+-----------+-----+
only showing top 30 rows


=== Pic de visionnage (semaine la plus active) ===
+----------+----------+-----+
|watch_year|watch_week|count|
+----------+----------+-----+
|      2023|         8|  109|
|      2023|         9|   94|
|      2025|         2|   94|
|      2023|        12|   73|
|    

## 5. Schéma final

In [10]:
# Sélection colonnes finales
df_final = df.select(
    "raw_title",
    "show_title",
    "season",
    "episode_title",
    "content_type",
    "watch_date",
    "watch_year",
    "watch_month",
    "watch_weekday",
    "watch_week",
    "interaction_weight"
)

print(f"Lignes finales : {df_final.count():,}")
df_final.printSchema()
df_final.show(5, truncate=50)

Lignes finales : 4,288
root
 |-- raw_title: string (nullable = true)
 |-- show_title: string (nullable = true)
 |-- season: integer (nullable = true)
 |-- episode_title: string (nullable = true)
 |-- content_type: string (nullable = false)
 |-- watch_date: date (nullable = true)
 |-- watch_year: integer (nullable = true)
 |-- watch_month: string (nullable = true)
 |-- watch_weekday: integer (nullable = true)
 |-- watch_week: integer (nullable = true)
 |-- interaction_weight: double (nullable = false)

+-----------------------------------+----------------------------+------+-------------+------------+----------+----------+-----------+-------------+----------+------------------+
|                          raw_title|                  show_title|season|episode_title|content_type|watch_date|watch_year|watch_month|watch_weekday|watch_week|interaction_weight|
+-----------------------------------+----------------------------+------+-------------+------------+----------+----------+-----------+-

## 6. Écriture Parquet

In [11]:
df_final.write \
    .mode("overwrite") \
    .parquet(PARQUET_OUT)

print(f"✓ Parquet écrit : {PARQUET_OUT}")

# Vérification lecture
df_check = spark.read.parquet(PARQUET_OUT)
print(f"✓ Vérification lecture : {df_check.count():,} lignes")
df_check.show(3, truncate=50)

✓ Parquet écrit : ../../data/parquet/netflix_views.parquet
✓ Vérification lecture : 4,288 lignes
+-----------------------------------+----------------------------+------+-------------+------------+----------+----------+-----------+-------------+----------+------------------+
|                          raw_title|                  show_title|season|episode_title|content_type|watch_date|watch_year|watch_month|watch_weekday|watch_week|interaction_weight|
+-----------------------------------+----------------------------+------+-------------+------------+----------+----------+-----------+-------------+----------+------------------+
|Black Mirror: Saison 4: Hang the DJ|                Black Mirror|     4|  Hang the DJ|      series|2026-03-26|      2026|    2026-03|            5|        13|               1.0|
|       Derrière nos écrans de fumée|Derrière nos écrans de fumée|  NULL|         NULL|       movie|2026-03-26|      2026|    2026-03|            5|        13|               1.0|
|       

## 7. Résumé

| Métrique | Valeur |
|---|---|
| Source | `NetflixViewingHistory.csv` |
| Lignes brutes | ~4 288 |
| Output | `netflix_views.parquet` |
| Usage ML | Matrice interactions ALS (axe 2) |
| Usage clustering | Activité par jour/semaine (axe 3) |

**Prochaine étape** : Enrichir avec l'API TMDB pour obtenir les genres (`Action`, `Drame`, `Animation`...) — optionnel mais utile pour le clustering thématique.

In [12]:
spark.stop()